# Dataset Logging: recording *which data + which features* produced a model

This is an **advanced** tutorial. It assumes you've worked through the tracking notebooks (`b_`–`e_`) and ideally the evaluation/registry/serving ones (`f_`–`h_`). It is also a **standalone split-out** of the capstone (`k_`): the capstone *uses* dataset logging at its feature-engineering step and cross-links back here for the mechanics.

So far every notebook has logged **params** (the knobs you turned) and **metrics** (how well it scored). Those answer *"what configuration produced this model?"* — but not *"what **data** produced it?"* As long as you train directly on a fixed public dataset, that gap is easy to ignore: "it's California housing, obviously."

The moment you do **feature engineering**, the gap opens. The columns the model actually trains on are no longer the raw columns — they're a *derived* feature set. Now "which data + which features produced this model?" is **not answerable** from params and metrics alone. The usual fallbacks are a sentence in a README, a hard-coded path, or "ask whoever wrote it" — none of which travel with the run.

`mlflow.data` + `mlflow.log_input(...)` close that gap: they attach **dataset lineage** (a name, a source pointer, a schema, and a content fingerprint) to the run, so a reader months later can reconstruct or audit exactly what went in.

## Source material

- **[Dataset tracking guide](https://mlflow.org/docs/latest/ml/dataset/)** — the top-level "what/why" for `mlflow.data`.
- **[`mlflow.data` API reference](https://mlflow.org/docs/latest/api_reference/python_api/mlflow.data.html)** — `from_pandas`, `from_numpy`, `from_spark`, the `Dataset`/`DatasetSource` objects, and `MetaDataset`.

This notebook has no single upstream counterpart — the official docs describe the API in reference form; here we motivate it with a concrete raw-vs-engineered scenario on California housing.

## What `mlflow.data` records, and what it replaces

Two objects do the work:

| Object | Built with | What it carries |
|---|---|---|
| **`Dataset`** | `mlflow.data.from_pandas(df, source=..., name=..., targets=...)` (also `from_numpy`, `from_spark`, …) | A **`schema`** (column names + types), a **`profile`** (row/element counts), a **`digest`** (a content fingerprint), and a **`source`** (a lineage pointer — a file path, URL, Delta table, …). |
| **The link to a run** | `mlflow.log_input(dataset, context=..., tags=...)` | Attaches the dataset to the active run under a free-form **`context`** label (`"raw_source"`, `"training"`, `"validation"`, `"testing"`, `"evaluation"`). **Multiple `log_input` calls per run are normal** — one per dataset. |

What it replaces:

| Without dataset logging | With `log_input` |
|---|---|
| "Which CSV was this? Let me check the README… which is stale." | The `source` pointer is on the run. |
| "Did run A and run B train on the same data?" → diff two files by hand | Compare the two runs' **digests**. |
| "These are *engineered* features — where's the recipe?" → ask the author | A named, fingerprinted engineered dataset (`ca-housing-engineered-v1`) sits on the run with its own source. |

**Cost note up front:** MLflow logs the dataset's **metadata + source reference + digest — not the full data**. That keeps runs cheap, but it means re-materializing the data later goes *through the source* (more on this in the storage gotcha below).

## Prerequisites

**No new dependencies** — `mlflow` and `pandas` (plus `scikit-learn`, used for a throwaway model) are already in the project.

### Start the tracking server

Before running any cell that calls `mlflow.set_tracking_uri(...)`, start the MLflow server **in a separate terminal from the repo root** (same convention as `f_`/`g_`/`h_`):

```bash
mlflow ui --host 127.0.0.1 --port 5001
```

Starting from the repo root keeps per-developer runtime state (`mlflow.db`, `mlartifacts/`) in a predictable place. The advanced notebooks all use port **5001**.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.datasets import fetch_california_housing
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split

import mlflow
from mlflow.exceptions import RestException

HOST, PORT = "127.0.0.1", 5001
mlflow.set_tracking_uri(f"http://{HOST}:{PORT}")

# Idempotent experiment creation (see principle 7 of the mlflow-tutorial-improve
# skill). Re-running this cell is a no-op.
experiment_name = "Dataset Logging"
try:
    mlflow.create_experiment(name=experiment_name)
except RestException as e:
    if "RESOURCE_ALREADY_EXISTS" not in str(e):
        raise
mlflow.set_experiment(experiment_name)

<Experiment: artifact_location='mlflow-artifacts:/5', creation_time=1780396495277, experiment_id='5', last_update_time=1780396495277, lifecycle_stage='active', name='Dataset Logging', tags={}, trace_location=None, workspace='default'>

## Step 1: A raw dataset object — with a *real* source

We load California housing and immediately write it to a CSV. That CSV is the dataset's **`source`**: a real, re-loadable pointer (not a throwaway string), which is what makes the storage/reload story at the end actually work.

`fetch_california_housing(as_frame=True).frame` gives the 8 features **plus** the target column `MedHouseVal` in one DataFrame — convenient, because `from_pandas(..., targets="MedHouseVal")` then knows which column is the label.

In [2]:
DEMO_DIR = Path("_dataset_demo")
DEMO_DIR.mkdir(exist_ok=True)
RAW_CSV = DEMO_DIR / "ca_housing_raw.csv"

raw_df = fetch_california_housing(as_frame=True).frame  # 8 features + "MedHouseVal"
raw_df.to_csv(RAW_CSV, index=False)                     # a real, re-loadable source on disk

print("wrote", RAW_CSV.resolve())
print("shape", raw_df.shape)
raw_df.head(3)

wrote /home/targ/venv_projects/teach-mlflow/src/ml/_dataset_demo/ca_housing_raw.csv
shape (20640, 9)


,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude,MedHouseVal
0,8.3252,41.0,6.984127,1.023810,322.0,2.555556,37.88,-122.23,4.526
1,8.3014,21.0,6.238137,0.971880,2401.0,2.109842,37.86,-122.22,3.585
2,7.2574,52.0,8.288136,1.073446,496.0,2.802260,37.85,-122.24,3.521


In [3]:
raw_ds = mlflow.data.from_pandas(
    raw_df,
    source=str(RAW_CSV),       # lineage pointer: a file we can reload later
    name="ca-housing-raw",
    targets="MedHouseVal",
)

print("name   :", raw_ds.name)
print("digest :", raw_ds.digest)
print("source :", raw_ds.source)
print("profile:", raw_ds.profile)
print("schema :", raw_ds.schema)

name   : ca-housing-raw
digest : c25ff0f6
source : <mlflow.data.artifact_dataset_sources.LocalArtifactDatasetSource object at 0x7df59c62e120>
profile: {'num_rows': 20640, 'num_elements': 185760}
schema : ['MedInc': double (required), 'HouseAge': double (required), 'AveRooms': double (required), 'AveBedrms': double (required), 'Population': double (required), 'AveOccup': double (required), 'Latitude': double (required), 'Longitude': double (required), 'MedHouseVal': double (required)]


/home/targ/venv_projects/teach-mlflow/.venv/lib/python3.14/site-packages/mlflow/data/dataset_source_registry.py:148: UserWarning: The specified dataset source can be interpreted in multiple ways: LocalArtifactDatasetSource, LocalArtifactDatasetSource. MLflow will assume that this is a LocalArtifactDatasetSource source.
  return _dataset_source_registry.resolve(


## Step 2: Feature-engineer, then build a *second* dataset

Now the part that motivates the whole notebook: we derive new columns. The engineered frame has columns the raw data never had, so it deserves its **own** dataset object with its **own** name, source, and digest — `ca-housing-engineered-v1`. The `v1` is deliberate: when you change the recipe, you bump to `v2`, and the digest changes too, so runs are never silently comparing different feature sets.

In [4]:
def engineer(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out["rooms_per_person"] = out["AveRooms"] / out["AveOccup"]
    out["bedrooms_per_room"] = out["AveBedrms"] / out["AveRooms"]
    out["log_median_income"] = np.log1p(out["MedInc"])
    return out


ENG_CSV = DEMO_DIR / "ca_housing_engineered_v1.csv"
eng_df = engineer(raw_df)
eng_df.to_csv(ENG_CSV, index=False)

eng_ds = mlflow.data.from_pandas(
    eng_df,
    source=str(ENG_CSV),
    name="ca-housing-engineered-v1",
    targets="MedHouseVal",
)

print("raw feature count       :", raw_df.shape[1] - 1)
print("engineered feature count:", eng_df.shape[1] - 1)
print("engineered digest       :", eng_ds.digest)
print("new columns             :", [c for c in eng_df.columns if c not in raw_df.columns])

raw feature count       : 8
engineered feature count: 11
engineered digest       : fc435491
new columns             : ['rooms_per_person', 'bedrooms_per_room', 'log_median_income']


/home/targ/venv_projects/teach-mlflow/.venv/lib/python3.14/site-packages/mlflow/data/dataset_source_registry.py:148: UserWarning: The specified dataset source can be interpreted in multiple ways: LocalArtifactDatasetSource, LocalArtifactDatasetSource. MLflow will assume that this is a LocalArtifactDatasetSource source.
  return _dataset_source_registry.resolve(


## Step 3: Log *both* datasets on the training run

This is the teaching core. One run, **two** `log_input` calls:

- the **raw source** under `context="raw_source"` — provenance ("here's where this started"),
- the **engineered feature set** under `context="training"` — what the model actually saw, tagged with the feature-set version and feature count.

Then we train a throwaway model on the engineered features, purely so the run links *data → model* the way a real one would.

In [5]:
feature_cols = [c for c in eng_df.columns if c != "MedHouseVal"]
X_train, X_test, y_train, y_test = train_test_split(
    eng_df[feature_cols], eng_df["MedHouseVal"], test_size=0.25, random_state=0
)

with mlflow.start_run(run_name="rf-on-engineered-v1") as run:
    mlflow.log_input(raw_ds, context="raw_source")
    mlflow.log_input(
        eng_ds,
        context="training",
        tags={"feature_set": "v1", "n_features": str(len(feature_cols))},
    )

    model = RandomForestRegressor(n_estimators=200, random_state=0).fit(X_train, y_train)
    mlflow.log_metric("test_r2", float(model.score(X_test, y_test)))
    mlflow.sklearn.log_model(model, name="model")
    run_id = run.info.run_id

print("run:", run_id)

2026/06/02 13:35:14 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


2026/06/02 13:35:16 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


🏃 View run rf-on-engineered-v1 at: http://127.0.0.1:5001/#/experiments/5/runs/1cd1172f3ec34d4ba4406b9f0900d85f
🧪 View experiment at: http://127.0.0.1:5001/#/experiments/5
run: 1cd1172f3ec34d4ba4406b9f0900d85f


In [6]:
# Read the inputs back off the run, the way a reviewer would.
logged = mlflow.get_run(run_id).inputs.dataset_inputs
for di in logged:
    ctx = {t.key: t.value for t in di.tags}.get("mlflow.data.context")
    print(f"context={ctx!r:14} name={di.dataset.name!r:28} digest={di.dataset.digest}")

context='raw_source'   name='ca-housing-raw'             digest=c25ff0f6
context='training'     name='ca-housing-engineered-v1'   digest=fc435491


## Step 4: Autolog vs manual — what each one captures

`b_`/`c_`/`e_` showed `autolog()` as the one-line alternative to manual logging, and sklearn/xgboost autolog log datasets too (`log_datasets=True` by default). So do you even need `log_input`?

You do — because **autolog logs whatever `X`/`y` reaches `.fit()`**, with a generic name and a weak (code) source. It does not know the frame came from a file, nor that it's "engineered v1". Watch the contrast: train a `Pipeline` under autolog, then read its dataset input back.

In [7]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

mlflow.sklearn.autolog()  # log_datasets=True is the default
with mlflow.start_run(run_name="autolog-pipeline") as auto_run:
    pipe = Pipeline(
        [("scale", StandardScaler()),
         ("rf", RandomForestRegressor(n_estimators=50, random_state=0))]
    )
    pipe.fit(X_train, y_train)
    auto_run_id = auto_run.info.run_id
mlflow.sklearn.autolog(disable=True)

print("autolog run:", auto_run_id)

2026/06/02 13:35:22 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


2026/06/02 13:35:24 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


🏃 View run autolog-pipeline at: http://127.0.0.1:5001/#/experiments/5/runs/8cf1ef631daf4b9c9c6200c6c2f4c6b1
🧪 View experiment at: http://127.0.0.1:5001/#/experiments/5
autolog run: 8cf1ef631daf4b9c9c6200c6c2f4c6b1


In [8]:
def show_inputs(rid, label):
    print(f"--- {label} ({rid[:8]}) ---")
    for di in mlflow.get_run(rid).inputs.dataset_inputs:
        d = di.dataset
        print(f"  name={d.name!r:28} source_type={d.source_type!r:10} digest={d.digest}")


show_inputs(run_id, "manual log_input")
show_inputs(auto_run_id, "autolog")

--- manual log_input (1cd1172f) ---
  name='ca-housing-raw'             source_type='local'    digest=c25ff0f6
  name='ca-housing-engineered-v1'   source_type='local'    digest=fc435491
--- autolog (8cf1ef63) ---
  name='dataset'                    source_type='code'     digest=7b46c535


Notice the difference: the **manual** run carries `ca-housing-engineered-v1` with a file source; the **autolog** run carries a generic `dataset` whose source is "code" — useful as a digest for skew detection, but it can't tell you *what* the features are or *where* they came from.

**Takeaway:** autolog is convenient and worth leaving on; the manual `log_input` of the engineered frame is what records the transformed feature set *properly*. Do both — autolog for the free baseline, `log_input` for real lineage.

## Step 5: The digest is a *fingerprint*, not a hash — a gotcha worth knowing

The `digest` is a short content fingerprint (a few hex characters) computed from a **sample** of the rows plus the frame's shape and column names. It's exactly what you want for "did these two runs train on the same data?" and for catching train/serve skew — cheap and stable. It is **not** a cryptographic, whole-content hash.

In [9]:
a = mlflow.data.from_pandas(eng_df, source=str(ENG_CSV), name="a")
b = mlflow.data.from_pandas(eng_df.copy(), source=str(ENG_CSV), name="b")
print("identical content  -> same digest :", a.digest == b.digest, "|", a.digest)

tweaked = eng_df.copy()
tweaked.iloc[0, 0] = tweaked.iloc[0, 0] + 1.0  # change a single value
c = mlflow.data.from_pandas(tweaked, source=str(ENG_CSV), name="c")
print("one value changed  -> new digest  :", a.digest != c.digest, "|", c.digest)

identical content  -> same digest : True | fc435491
one value changed  -> new digest  : True | d935da61


/home/targ/venv_projects/teach-mlflow/.venv/lib/python3.14/site-packages/mlflow/data/dataset_source_registry.py:148: UserWarning: The specified dataset source can be interpreted in multiple ways: LocalArtifactDatasetSource, LocalArtifactDatasetSource. MLflow will assume that this is a LocalArtifactDatasetSource source.
  return _dataset_source_registry.resolve(
/home/targ/venv_projects/teach-mlflow/.venv/lib/python3.14/site-packages/mlflow/data/dataset_source_registry.py:148: UserWarning: The specified dataset source can be interpreted in multiple ways: LocalArtifactDatasetSource, LocalArtifactDatasetSource. MLflow will assume that this is a LocalArtifactDatasetSource source.
  return _dataset_source_registry.resolve(
/home/targ/venv_projects/teach-mlflow/.venv/lib/python3.14/site-packages/mlflow/data/dataset_source_registry.py:148: UserWarning: The specified dataset source can be interpreted in multiple ways: LocalArtifactDatasetSource, LocalArtifactDatasetSource. MLflow will assume t

Same bytes → same digest; one changed value → a new digest. The caveat to remember: because only a **sample** of rows (the first ~10 000) feeds the fingerprint, a change *beyond* that window on a very large frame may not move the digest. Treat it as a fast equality check, not a guarantee.

## Step 6: Storage gotcha — MLflow stores the *pointer*, so reload via the source

Because only metadata + source + digest are stored, re-materializing the data goes **through the source**. This is why a *real* `source` matters: pass a throwaway string and you log a dataset you can't reload. With our file source, `source.load()` brings it back.

In [10]:
print("logged source:", eng_ds.source)

local_path = eng_ds.source.load()       # returns the local path for a file source
print("source.load() ->", local_path)

reloaded = pd.read_csv(local_path)
print("reloaded shape:", reloaded.shape)

logged source: <mlflow.data.artifact_dataset_sources.LocalArtifactDatasetSource object at 0x7df4e02a1450>
source.load() -> /home/targ/venv_projects/teach-mlflow/src/ml/_dataset_demo/ca_housing_engineered_v1.csv
reloaded shape: (20640, 12)


If you only have metadata and *intend* to log metadata-only (no re-loadable source), use **`mlflow.data.MetaDataset`** — the explicit "I'm recording that this data existed, not how to reload it" variant. Reaching for `from_pandas` with a fake `source` just to silence the API is the anti-pattern this avoids.

## Step 7: `evaluate(data=...)` records datasets differently (a nuance)

`f_` used `mlflow.models.evaluate(...)`. It accepts an `mlflow.data.Dataset` via `data=`, but it records that dataset to the run's **`mlflow.datasets` tag** rather than calling `log_input`. Practical consequence: if you want the eval set to appear in the run's **Datasets** panel (not just buried in a tag), **also** `log_input(eval_ds, context="evaluation")` yourself. The two mechanisms don't overlap — tag for evaluate's bookkeeping, `log_input` for the UI panel and the run's formal inputs.

## Step 8: Viewing it in the UI

Open <http://127.0.0.1:5001/> → **Dataset Logging** → the `rf-on-engineered-v1` run.

- The **Datasets** panel (run Overview) lists **two** entries: `ca-housing-raw` (context `raw_source`) and `ca-housing-engineered-v1` (context `training`), each with its schema, profile, digest, and source.
- The engineered entry shows the `feature_set=v1` / `n_features` tags you attached.
- Open the `autolog-pipeline` run next to it and compare: one generic `dataset` entry, code source — the contrast from Step 4, now visible.

That side-by-side is the whole point: a reviewer can see *what data and which feature set* produced the model without reading your code.

## Next steps

- **[Dataset tracking guide](https://mlflow.org/docs/latest/ml/dataset/)** and the **[`mlflow.data` API](https://mlflow.org/docs/latest/api_reference/python_api/mlflow.data.html)** — `from_numpy`, `from_spark`, `from_huggingface`, the `DatasetSource` types, and `MetaDataset`.
- **The capstone (`k_`)** uses exactly this pattern at its feature-engineering step, then carries the engineered dataset through evaluate → register → serve — lineage that survives the whole lifecycle.
- **Versioning feature sets.** Bump `v1` → `v2` when the recipe changes and the digest moves with it; filter runs by the `feature_set` tag to compare like with like.